In [1]:
# --- Setup: load API key and create ENTSO-E client ---
from dotenv import load_dotenv
import os
from entsoe import EntsoePandasClient
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pandas import Timestamp

load_dotenv()
api_key = os.getenv("ENTSOE_API_KEY")
client = EntsoePandasClient(api_key=api_key)

# Full year 2024, Austria timezone
start = Timestamp("2024-01-01", tz="Europe/Vienna")
end   = Timestamp("2024-12-31", tz="Europe/Vienna")

print("Client ready. Date range:", start.date(), "to", end.date())

Client ready. Date range: 2024-01-01 to 2024-12-31


In [2]:
# --- 1. Pull actual wind generation for Austria ---
# query_generation returns a DataFrame with one column per generation type.
# We extract the 'Wind Onshore' column.
gen_raw = client.query_generation("AT", start=start, end=end)
print("Generation columns:", gen_raw.columns.tolist())

# Handle MultiIndex columns if present (some entsoe-py versions return them)
if isinstance(gen_raw.columns, pd.MultiIndex):
    gen_raw.columns = [" ".join(c).strip() for c in gen_raw.columns]

# Find the wind onshore column (name may vary slightly)
wind_col = [c for c in gen_raw.columns if "Wind Onshore" in c][0]
actual = gen_raw[wind_col].copy()
actual.name = "actual"
print(f"Actual wind onshore column: '{wind_col}' — {len(actual)} rows")

Generation columns: [('Biomass', 'Actual Aggregated'), ('Biomass', 'Actual Consumption'), ('Fossil Gas', 'Actual Aggregated'), ('Fossil Gas', 'Actual Consumption'), ('Fossil Hard coal', 'Actual Aggregated'), ('Fossil Hard coal', 'Actual Consumption'), ('Fossil Oil', 'Actual Aggregated'), ('Fossil Oil', 'Actual Consumption'), ('Geothermal', 'Actual Aggregated'), ('Geothermal', 'Actual Consumption'), ('Hydro Pumped Storage', 'Actual Aggregated'), ('Hydro Pumped Storage', 'Actual Consumption'), ('Hydro Run-of-river and poundage', 'Actual Aggregated'), ('Hydro Run-of-river and poundage', 'Actual Consumption'), ('Hydro Water Reservoir', 'Actual Aggregated'), ('Hydro Water Reservoir', 'Actual Consumption'), ('Other', 'Actual Aggregated'), ('Other', 'Actual Consumption'), ('Other renewable', 'Actual Aggregated'), ('Other renewable', 'Actual Consumption'), ('Solar', 'Actual Aggregated'), ('Solar', 'Actual Consumption'), ('Waste', 'Actual Aggregated'), ('Waste', 'Actual Consumption'), ('Wind On

In [3]:
# --- 2. Pull wind + solar forecasts for Austria ---
# query_wind_and_solar_forecast returns a DataFrame with Wind and Solar columns.
# We extract the wind forecast column.
fc_raw = client.query_wind_and_solar_forecast("AT", start=start, end=end)
print("Forecast columns:", fc_raw.columns.tolist())

if isinstance(fc_raw.columns, pd.MultiIndex):
    fc_raw.columns = [" ".join(c).strip() for c in fc_raw.columns]

fc_col = [c for c in fc_raw.columns if "Wind Onshore" in c][0]
forecast = fc_raw[fc_col].copy()
forecast.name = "forecast"
print(f"Forecast wind column: '{fc_col}' — {len(forecast)} rows")

Forecast columns: ['Solar', 'Wind Onshore']
Forecast wind column: 'Wind Onshore' — 35040 rows


In [4]:
# --- 3. Align both series to hourly timestamps and compute forecast error ---
# Resample to hourly means to ensure both series share the same index.
actual_h   = actual.resample("1h").mean()
forecast_h = forecast.resample("1h").mean()

# Inner join: only keep timestamps present in both series
df = pd.DataFrame({"actual": actual_h, "forecast": forecast_h}).dropna()

# Forecast error: positive = overforecast, negative = underforecast
df["error"] = df["forecast"] - df["actual"]

print(f"Aligned dataset: {len(df)} hourly observations")
print(df.describe().round(1))

Aligned dataset: 8760 hourly observations
       actual  forecast   error
count  8760.0    8760.0  8760.0
mean   1069.1    1052.2   -16.8
std     945.4     883.6   390.7
min       4.0      16.0 -2032.0
25%     257.0     305.0  -196.0
50%     763.0     789.0     1.0
75%    1721.2    1636.0   174.0
max    3498.0    3398.0  1949.0


In [5]:
# --- Chart 1: Full year — actual wind vs forecast overlaid ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["actual"],
    name="Actual",
    line=dict(color="steelblue", width=1),
    opacity=0.85
))
fig.add_trace(go.Scatter(
    x=df.index, y=df["forecast"],
    name="Forecast",
    line=dict(color="orange", width=1),
    opacity=0.75
))

fig.update_layout(
    title="AT Wind Onshore 2024 — Actual vs Forecast (MW)",
    xaxis_title="Date",
    yaxis_title="MW",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
    template="plotly_white"
)
fig.show()

In [6]:
# --- Chart 2: Full year forecast error ---
# Color positive errors (overforecast) red, negative (underforecast) blue.
colors = ["crimson" if e > 0 else "steelblue" for e in df["error"]]

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=df.index,
    y=df["error"],
    marker_color=colors,
    name="Forecast Error"
))
fig2.add_hline(y=0, line_dash="dash", line_color="black", line_width=1)

fig2.update_layout(
    title="AT Wind Forecast Error 2024 (Forecast − Actual, MW)",
    xaxis_title="Date",
    yaxis_title="Error (MW)",
    hovermode="x unified",
    template="plotly_white"
)
fig2.show()

In [7]:
# --- Chart 3: Monthly bar chart of MAE and RMSE ---
df["month"] = df.index.month
monthly_metrics = df.groupby("month")["error"].agg(
    MAE=lambda x: np.abs(x).mean(),
    RMSE=lambda x: np.sqrt((x**2).mean())
).reset_index()

month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]
monthly_metrics["month_name"] = monthly_metrics["month"].apply(lambda m: month_labels[m-1])

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=monthly_metrics["month_name"], y=monthly_metrics["MAE"],
    name="MAE", marker_color="steelblue"
))
fig3.add_trace(go.Bar(
    x=monthly_metrics["month_name"], y=monthly_metrics["RMSE"],
    name="RMSE", marker_color="coral"
))

fig3.update_layout(
    title="Monthly Wind Forecast Error — MAE & RMSE (MW)",
    xaxis_title="Month",
    yaxis_title="Error (MW)",
    barmode="group",
    template="plotly_white"
)
fig3.show()

In [8]:
# --- Chart 4: Histogram of all forecast errors ---
fig4 = go.Figure()
fig4.add_trace(go.Histogram(
    x=df["error"],
    nbinsx=80,
    marker_color="steelblue",
    opacity=0.8,
    name="Error distribution"
))
fig4.add_vline(x=0, line_dash="dash", line_color="black", line_width=1)
fig4.add_vline(x=df["error"].mean(), line_dash="dot", line_color="crimson",
               annotation_text=f"Mean {df['error'].mean():.0f} MW",
               annotation_position="top right")

fig4.update_layout(
    title="Distribution of Wind Forecast Errors 2024 (MW)",
    xaxis_title="Forecast Error (MW)  [positive = overforecast]",
    yaxis_title="Count (hours)",
    template="plotly_white"
)
fig4.show()

In [9]:
# --- Chart 5: 20 worst forecast error hours ---
# "Worst" = largest absolute error regardless of sign.
worst20 = df["error"].abs().nlargest(20).index
worst_df = df.loc[worst20, ["actual", "forecast", "error"]].copy()
worst_df = worst_df.sort_values("error", key=abs, ascending=True)  # sort for horizontal bar

labels = worst_df.index.strftime("%Y-%m-%d %H:%M")
bar_colors = ["crimson" if e > 0 else "steelblue" for e in worst_df["error"]]

fig5 = go.Figure()
fig5.add_trace(go.Bar(
    x=worst_df["error"],
    y=labels,
    orientation="h",
    marker_color=bar_colors,
    text=worst_df["error"].round(0).astype(int).astype(str) + " MW",
    textposition="outside",
    name="Error"
))
fig5.add_vline(x=0, line_dash="dash", line_color="black", line_width=1)

fig5.update_layout(
    title="20 Worst Wind Forecast Error Hours 2024 (MW)",
    xaxis_title="Forecast Error (MW)  [positive = overforecast]",
    yaxis_title="Timestamp",
    height=600,
    template="plotly_white"
)
fig5.show()

# Also print as a table
print("\n20 Worst Forecast Error Hours:")
display_df = worst_df.copy()
display_df.index = display_df.index.strftime("%Y-%m-%d %H:%M %Z")
display_df.columns = ["Actual (MW)", "Forecast (MW)", "Error (MW)"]
display_df = display_df.sort_values("Error (MW)", key=abs, ascending=False)
print(display_df.round(0).to_string())


20 Worst Forecast Error Hours:
                       Actual (MW)  Forecast (MW)  Error (MW)
2024-03-11 23:00 CET        2410.0          378.0     -2032.0
2024-09-16 18:00 CEST        377.0         2326.0      1949.0
2024-08-08 16:00 CEST       2233.0          297.0     -1936.0
2024-12-23 18:00 CET        3192.0         1271.0     -1921.0
2024-08-02 22:00 CEST       2802.0          883.0     -1919.0
2024-12-23 19:00 CET        3216.0         1326.0     -1890.0
2024-09-16 17:00 CEST        650.0         2512.0      1862.0
2024-12-23 16:00 CET        3140.0         1286.0     -1854.0
2024-09-06 00:00 CEST       1236.0         3089.0      1853.0
2024-09-16 19:00 CEST        309.0         2130.0      1821.0
2024-07-17 15:00 CEST       2357.0          590.0     -1767.0
2024-12-23 17:00 CET        3010.0         1258.0     -1752.0
2024-12-23 15:00 CET        3149.0         1404.0     -1745.0
2024-12-21 13:00 CET        2445.0          708.0     -1737.0
2024-12-23 20:00 CET        3139.0    

In [10]:
# --- Summary table: per-month MAE, RMSE, Bias, MAPE ---
def mape(actual, error):
    # Avoid division by zero: exclude hours where actual == 0
    mask = actual != 0
    return np.abs(error[mask] / actual[mask]).mean() * 100

records = []
for m in range(1, 13):
    mdf = df[df["month"] == m]
    records.append({
        "Month": month_labels[m - 1],
        "MAE (MW)": np.abs(mdf["error"]).mean(),
        "RMSE (MW)": np.sqrt((mdf["error"] ** 2).mean()),
        "Bias (MW)": mdf["error"].mean(),          # mean error: + = systematic overforecast
        "MAPE (%)": mape(mdf["actual"], mdf["error"])
    })

summary = pd.DataFrame(records).set_index("Month")

# Annual row
summary.loc["FULL YEAR"] = [
    np.abs(df["error"]).mean(),
    np.sqrt((df["error"] ** 2).mean()),
    df["error"].mean(),
    mape(df["actual"], df["error"])
]

print("\nWind Forecast Error Summary — Austria 2024")
print(summary.round(1).to_string())


Wind Forecast Error Summary — Austria 2024
           MAE (MW)  RMSE (MW)  Bias (MW)  MAPE (%)
Month                                              
Jan           320.0      435.9      -24.0      51.3
Feb           285.7      374.9      -33.7      55.5
Mar           300.7      424.8     -102.1      55.7
Apr           275.4      373.8       76.3      42.0
May           222.1      314.2      -24.2      72.5
Jun           217.9      299.1       -3.7     124.9
Jul           294.8      426.3      -25.2     101.8
Aug           219.1      323.4      -10.1     109.1
Sep           260.4      392.6       35.6      63.4
Oct           255.1      375.1       36.4      73.2
Nov           301.3      427.2      -40.3     136.0
Dec           338.8      482.1      -86.4      61.0
FULL YEAR     274.2      391.0      -16.8      78.9


## December 1st Deep-Dive

In [11]:
# --- Filter to December 1st and fetch DA + imbalance prices ---
dec1_start = Timestamp("2024-03-11", tz="Europe/Vienna")
dec1_end   = Timestamp("2024-03-12", tz="Europe/Vienna")

df_dec1 = df.loc["2024-03-11"]
df_dec1 = df_dec1.copy()
df_dec1["abs_error"] = df_dec1["error"].abs()
df_dec1["rel_error"] = (df_dec1["abs_error"] / df_dec1["actual"]) * 100

# Day-ahead prices
da_prices = client.query_day_ahead_prices("AT", start=dec1_start, end=dec1_end)
da_prices = da_prices.resample("1h").mean()
da_prices.name = "da_price"

# Imbalance prices
imb_raw = client.query_imbalance_prices("AT", start=dec1_start, end=dec1_end)
print("Imbalance columns:", imb_raw.columns.tolist())
imb_prices = imb_raw.resample("1h").mean()

print(f"Dec 1 wind rows:      {len(df_dec1)}")
print(f"Dec 1 DA price rows:  {len(da_prices)}")
print(f"Dec 1 imb price rows: {len(imb_prices)}")
print(df_dec1[["actual", "forecast", "error", "abs_error", "rel_error"]].round(1))

Imbalance columns: ['Long', 'Short']
Dec 1 wind rows:      24
Dec 1 DA price rows:  25
Dec 1 imb price rows: 24
                           actual  forecast   error  abs_error  rel_error
2024-03-11 00:00:00+01:00  2320.0    2731.0   411.0      411.0       17.7
2024-03-11 01:00:00+01:00  2820.0    2638.0  -182.0      182.0        6.5
2024-03-11 02:00:00+01:00  2763.0    2460.0  -303.0      303.0       11.0
2024-03-11 03:00:00+01:00  1971.0    2171.0   200.0      200.0       10.1
2024-03-11 04:00:00+01:00  1228.0    1758.0   530.0      530.0       43.2
2024-03-11 05:00:00+01:00  1046.0    1300.0   254.0      254.0       24.3
2024-03-11 06:00:00+01:00   683.0     958.0   275.0      275.0       40.3
2024-03-11 07:00:00+01:00   323.0     723.0   400.0      400.0      123.8
2024-03-11 08:00:00+01:00   134.0     517.0   383.0      383.0      285.8
2024-03-11 09:00:00+01:00    66.0     375.0   309.0      309.0      468.2
2024-03-11 10:00:00+01:00    46.0     296.0   250.0      250.0      543.5


In [12]:
# --- Chart A: Dec 1 — Hourly Actual vs Forecast ---
fig_a = go.Figure()

fig_a.add_trace(go.Scatter(
    x=df_dec1.index, y=df_dec1["actual"],
    name="Actual",
    mode="lines+markers",
    line=dict(color="steelblue", width=2),
    marker=dict(size=6)
))
fig_a.add_trace(go.Scatter(
    x=df_dec1.index, y=df_dec1["forecast"],
    name="Forecast",
    mode="lines+markers",
    line=dict(color="orange", width=2, dash="dash"),
    marker=dict(size=6)
))

fig_a.update_layout(
    title="December 1, 2024 — Wind Onshore: Actual vs Forecast (MW)",
    xaxis_title="Hour",
    yaxis_title="MW",
    xaxis=dict(tickformat="%H:%M"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="x unified",
    template="plotly_white"
)
fig_a.show()

In [13]:
# --- Chart B: Dec 1 — Absolute error, relative error (%), DA price, imbalance price ---
from plotly.subplots import make_subplots

fig_b = make_subplots(
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=(
        "Absolute Forecast Error (MW)",
        "Relative Forecast Error (%)",
        "Day-Ahead Price (€/MWh)",
        "Imbalance Price (€/MWh)"
    )
)

bar_colors = ["crimson" if e > 0 else "steelblue" for e in df_dec1["error"]]

# Row 1: Signed error bars
fig_b.add_trace(go.Bar(
    x=df_dec1.index, y=df_dec1["error"],
    marker_color=bar_colors,
    name="Abs Error (MW)"
), row=1, col=1)
fig_b.add_hline(y=0, line_dash="dash", line_color="black", line_width=1, row=1, col=1)

# Row 2: Relative error (%)
fig_b.add_trace(go.Scatter(
    x=df_dec1.index, y=df_dec1["rel_error"],
    mode="lines+markers",
    line=dict(color="purple", width=2),
    marker=dict(size=6),
    name="Rel. Error (%)"
), row=2, col=1)

# Row 3: DA price
fig_b.add_trace(go.Scatter(
    x=da_prices.index, y=da_prices.values,
    mode="lines+markers",
    line=dict(color="darkgreen", width=2),
    marker=dict(size=6),
    name="DA Price (€/MWh)"
), row=3, col=1)

# Row 4: Imbalance prices — plot all columns returned
for col in imb_prices.columns:
    fig_b.add_trace(go.Scatter(
        x=imb_prices.index, y=imb_prices[col],
        mode="lines+markers",
        marker=dict(size=5),
        name=f"Imb: {col}"
    ), row=4, col=1)

fig_b.update_xaxes(tickformat="%H:%M", row=4, col=1)
fig_b.update_yaxes(title_text="Error (MW)", row=1, col=1)
fig_b.update_yaxes(title_text="Rel. Error (%)", row=2, col=1)
fig_b.update_yaxes(title_text="€/MWh", row=3, col=1)
fig_b.update_yaxes(title_text="€/MWh", row=4, col=1)

fig_b.update_layout(
    title="December 1, 2024 — Forecast Error, DA Price & Imbalance Price",
    hovermode="x unified",
    template="plotly_white",
    height=950,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
fig_b.show()

## Imbalance Price Analysis — Austria 2024

In [14]:
# --- Fetch full-year imbalance prices for Austria 2024 ---
imb_year_raw = client.query_imbalance_prices("AT", start=start, end=end)
print("Imbalance columns:", imb_year_raw.columns.tolist())

# Resample to hourly means
imb_year = imb_year_raw.resample("1h").mean()
print(f"Rows: {len(imb_year)}")

# --- Statistics ---
stats = imb_year.describe().loc[["min", "mean", "50%", "max", "std"]]
stats.index = ["Min", "Mean", "Median", "Max", "Std Dev"]
print("\nImbalance Price Statistics 2024 (€/MWh)")
print(stats.round(2).to_string())

Imbalance columns: ['Long', 'Short']
Rows: 8760

Imbalance Price Statistics 2024 (€/MWh)
             Long     Short
Min     -11214.38 -11214.38
Mean        72.25     72.25
Median      75.74     75.74
Max       4502.72   4502.72
Std Dev    273.26    273.26


In [15]:
# --- Chart: Hourly imbalance price overlaid by month (30-day x-axis) ---
# X-axis = fractional day-of-month (0–31), one trace per month, each a different color.

month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]
colors_12 = [
    "#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b",
    "#e377c2","#7f7f7f","#bcbd22","#17becf","#aec7e8","#ffbb78"
]

# Use the first column (or "Long" if available) as the main imbalance price series
price_col = imb_year.columns[0]
imb_series = imb_year[price_col].copy()

fig_imb = go.Figure()

for m in range(1, 13):
    month_data = imb_series[imb_series.index.month == m]
    # Fractional day-of-month as x (hour 0 of day 1 = 0.0, hour 23 of day 31 = 30.958...)
    x_vals = (month_data.index.day - 1) + month_data.index.hour / 24

    fig_imb.add_trace(go.Scatter(
        x=x_vals,
        y=month_data.values,
        mode="lines",
        name=month_labels[m - 1],
        line=dict(color=colors_12[m - 1], width=1),
        opacity=0.8
    ))

fig_imb.update_layout(
    title="AT Imbalance Price 2024 — Hourly by Month (€/MWh)",
    xaxis_title="Day of Month",
    yaxis_title="€/MWh",
    xaxis=dict(
        tickmode="array",
        tickvals=list(range(0, 31, 5)),
        ticktext=[f"Day {d+1}" for d in range(0, 31, 5)]
    ),
    legend=dict(title="Month", orientation="v"),
    hovermode="x unified",
    template="plotly_white",
    height=500
)
fig_imb.show()